# Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from keras import layers
import sklearn
import glob
import json
from sklearn.metrics import roc_curve,auc
import os
from keras.optimizers import Adam
from keras.saving import register_keras_serializable
import pandas as pd

# Student Functions

First, we define the student architectures.

In [ ]:
def get_student_large():
    '''
    Creates the NN-large student architecture.

    Returns: (NN_large_model, "student_large")
    '''
    latent_dim=8
    num_filters=16
    print("Creating 3 layer model with latent dim 2x2x", latent_dim)
    model = keras.models.Sequential()
    model.add(layers.Input((24,24,1)))
    model.add(layers.Conv2D(num_filters, (4,4),2,activation="relu"))
    model.add(layers.Conv2D(num_filters, (3,3),2,activation="relu"))
    model.add(layers.Conv2D(latent_dim, (3,3),2,activation="relu"))
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    model.summary()
    name = "student_large"
    print("name: ",name)
    return model,name

def get_student_medium():
    '''
    Creates the NN-medium student architecture.

    Returns: (NN_medium_model, "student_medium")
    '''
    print("Creating medium student")
    model = keras.models.Sequential()
    model.add(layers.Input((24,24,1)))
    model.add(layers.Flatten())
    model.add(layers.Dense(32,activation="relu"))
    model.add(layers.Dense(16,activation="relu"))
    model.add(layers.Dense(1))
    model.summary()
    name = "student_medium"
    print("name: ",name)
    return model,name

def get_student_small():
    '''
    Creates the NN-small student architecture.

    Returns: (NN_small_model, "student_small")
    '''
    print("Creating small student")
    model = keras.models.Sequential()
    model.add(layers.Input(shape=(24, 24, 1)))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))  # Reduces 24x24 → 12x12
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    model.summary()
    name = "student_small"
    print("name: ",name)
    return model,name

def get_student_tiny():
    '''
    Creates the NN-tiny student architecture.

    Returns: (NN_tiny_model, "student_tiny")
    '''
    model = keras.models.Sequential()
    model.add(layers.Input(shape=(24, 24, 1)))
    model.add(layers.MaxPooling2D(pool_size=(3, 3),strides=3,padding='valid'))  # Reduces 24x24 → 8x8x1
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    model.summary()
    name = "student_tiny"
    print("name: ",name)
    return model,name

def get_roc_data_student(model,background,signals):
    '''
    Uses the [model] to make predictions on the [background] and [signals] data.
    Then, produces a dictionary that contains the FPRs, TPRs, and thresholds for
    each signal against the background data as well as the scores for each class
    of data.

    The dictionary is of the following format:
        roc_data_dict = {'fprs' :
                            {'w' : [...] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                         'tprs' : 
                              {'w' : [...] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                         'thresholds' : 
                              {'w' : [...] , 'z' : [...], 't' : [...], 'combined_signal' : [...]},
                         'scores' : 
                              {'background' : [...], 'w' : [...] , 'z' : [...], 't' : [...], 'combined_signal' : [...]}
                         }
    Args:
        - model : trained keras model
        - background (np.array) : array of shape [N,24,24,1] of the background data
        - signals (dict of key (str) : value (np.array) : dictionary with keys labelling
                                        the signals 'w','z', and 't' and the values giving
                                        the signal data of shape [N,24,24,1]

    Returns:
        The roc_data_dict as described above.
    '''
    preds = model.predict(background,verbose=0)
    
    scores_dict = {}
    scores_dict['background'] = preds
    
    fprs = {}
    tprs = {}
    thresholds = {}

    for key in signals.keys():
        signal = signals[key]
        signal_preds = model.predict(signal,verbose=0)

        labels = np.concatenate((np.full(shape=signal_preds.shape[0],fill_value=1),np.full(shape=preds.shape[0],fill_value=0)))
        scores = np.concatenate((signal_preds, preds))
        fpr,tpr,threshold = roc_curve(y_true = labels,y_score=scores)

        fprs[key] = fpr
        tprs[key] = tpr
        thresholds[key] = threshold  
        scores_dict[key] = signal_preds

    background_scores = preds
    combined_signals = np.concatenate([scores_dict[signal] for signal in ['w','z','t']])
    
    combined_labels = np.concatenate((np.full(shape=combined_signals.shape[0],fill_value=1),np.full(shape=background_scores.shape[0],fill_value=0)))
    combined_scores = np.concatenate((combined_signals,background_scores))
    combined_fpr,combined_tpr,combined_threshold = roc_curve(y_true = combined_labels,y_score=combined_scores)
    fprs['combined_signal'] = combined_fpr
    tprs['combined_signal'] = combined_tpr
    thresholds['combined_signal'] = combined_threshold
    scores_dict['combined_signal']= combined_signals        

    return {'fprs' : fprs, "tprs" : tprs, "thresholds":thresholds,'scores':scores_dict}

def train_student_model(save_name,model,train_ds,val_ds,test,signals,epochs=300):
    '''
    Trains the [model] with [train_ds] as the training dataset and [val_ds] as the validation dataset.
    Calls get_roc_data_student with [test] as the background data and [signals] as the signal data
    dictionary to obtain the ROC curve data for evaluation of the models.

    Args:
        - save_name : str used while saving model and loss plot
        - model : the keras model to train
        - train_ds : the tf.data.Dataset for training
        - val_ds : the tf.data.Dataset for validation
        - test : the .npy array containing quark and gluon test images
        - signals : the dictionary containing keys 'w','z','t' and values of the corresponding
                    signal .npy arrays
        - epochs : the number of epochs to train

    Returns:
        The history object from tf.callbacks.History and the roc_data_dict from get_roc_data_student
    '''
    print("Starting training")
    t_1 = time.time()
    callback = keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                              patience=5, min_lr=1e-6)
    model_history = model.fit(x=train_ds,validation_data=val_ds,epochs=epochs,
                               callbacks=[callback,reduce_lr])
    t_2 = time.time()
    print('Training took {} minutes'.format((t_2-t_1)/60))
    
    print("Saving model as:", save_name)
    model.save(save_name + ".keras")
    
    plt.plot(model_history.history['loss'],label='loss')
    plt.plot(model_history.history['val_loss'],label='val_loss')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()
    plt.savefig(f"{save_name}_loss.png")
    plt.show()

    roc_data_student = get_roc_data_student(model,test,signals)    
    
    return model_history, roc_data_student

In [ ]:
def target_dictionary(model_path,data_path,beta,transformation=lambda x:x):
    '''
    Produces a dictionary with the five scores derived from the teacher model at [model_path] with
    data at [data_path] transformed with the given [transformation].

    Args:
        - model_path (str) : the path to the teacher .keras model
        - data_path (str) : the path to the data directory
        - beta (float) : the beta hyperparameter value used for the teacher at [model_path]
        - transformation : the data transformation used for data to train the teacher at [model_path]

    Returns:
        A dictionary with all five scores.
    '''
    model = keras.models.load_model(model_path)
    MSEs = []
    KLs = []
    mus = []
    sigmas = []
    MSE_KLs = []
    for i,file_name in enumerate(np.sort(glob.glob(data_path))):
        print(file_name)
        data = transformation(np.load(file_name))
        print("Data shape: ",data.shape)
        if i==0:
            plt.imshow(np.mean(data[0:1000],axis=(0)))
            plt.title("Averaged event")
            plt.colorbar()
            plt.show()
        preds = model.predict(data)
        
        outputs = preds['output']
       
        MSE = np.mean(np.square(data-outputs),axis=(1,2,3))
        #print("MSE shape",MSE.shape)
        MSEs.extend(MSE)
        
        mu = preds['latent_z_mean']
        log_sigma = preds['latent_z_log_var']

        mus.extend(np.sum(np.square(mu),axis=-1))
        # UPDATED SIGMAS DEFINITION
        #sigmas.extend(np.sum(np.exp(log_sigma)+1,axis=-1))
        sigmas.extend(np.mean((np.sqrt(np.exp(log_sigma)) - 1),axis=-1))
    
        KL = 0.5 * (np.sum(np.square(mu) + np.exp(log_sigma) - 1 - log_sigma,axis=-1))
        #print("KL shape",KL.shape)

        KLs.extend(KL)
        
        MSE_KL = (1-beta)*MSE + (beta)*KL
        MSE_KLs.extend(MSE_KL)
    data = {"MSE": MSEs, "KL":KLs, "mus": mus, "sigmas":sigmas,"MSE_KL": MSE_KLs}
    
    for key in data.keys():
        plt.hist(data[key],bins=50,histtype='step')
        plt.title(data_path)
        plt.xlabel(key)
        plt.ylabel("Events")
        plt.yscale('log')
        plt.show()
    return data

def get_train_test_val_target_dict(name,model_path,data_path,beta,transformation=lambda x:x):
    '''
    Creates a dictionary of 'train_targets', 'test_targets' and 'val_targets', which contains dictionaries
    of targets for each of the five anomaly scores from Equations (2)-(6) in the paper. The targets
    are produced from the model at [model_path] with the data transformed with the [transformation].

    The dictionary format follows:
        data_dict  = {'train_targets' : 
                        {'MSE' : [...], 'KL' : [...], 'mus' : [...], 'sigmas' : [], 'MSE_KL' : [...]}
                      'test_targets' :
                         {'MSE' : [...], ...                                                        }
                      'val_targets' : 
                         {'MSE' : [...], ...                                                        }
                    }
    Args:
        - name (str) : label for saving the dictionary as a .csv file
        - model_path (str) : path to the teacher model from which to calculate the target scores
        - data_path (str) : path to the directory with the data from which to calculate the target scores
        - beta (float) : the beta hyperparameter used to train the teacher model and which to use for the
                         calculation of the MSE+KL score
        - transformation : the transformation to apply to the data, must match the preprocessing used
                           for the data that trained the teacher model

    Returns the data_dict as described above and saves each train_targets, test_targets, and val_targets as
    .csv files.
    '''
    train = target_dictionary(model_path,data_path+"train/*",beta,transformation)
    test = target_dictionary(model_path,data_path+"test/*",beta,transformation)
    val = target_dictionary(model_path,data_path+"val/*",beta,transformation)
    data_dict = {"train_targets":train, "test_targets":test,"val_targets":val}
    save_targets_to_csv(name,data_dict)
    return data_dict

def save_targets_to_csv(name,data_dict):
    '''
    Save the 'train_targets', 'test_targets', and 'val_targets' data from data_dict
    as .csv files with the label from [name].

    Args:
        - name : the label used to save the file
        - data_dict : a dictionary with keys 'train_targets', 'test_targets', and 'val_targets'
                      and corresponding score information.
    '''
    df_train = pd.DataFrame.from_dict(data_dict["train_targets"])
    df_test = pd.DataFrame.from_dict(data_dict["test_targets"])
    df_val = pd.DataFrame.from_dict(data_dict["val_targets"])

    # Export the DataFrame to a CSV file
    df_train.to_csv(f"{name}_train_targets.csv", index=False)
    df_test.to_csv(f"{name}_test_targets.csv", index=False)
    df_val.to_csv(f"{name}_val_targets.csv", index=False)
    return

##################################################################################
# Functions to load the training data and targets as pairs for regression
##################################################################################
# The score targets are calculated on the data in a defined order. When creating
# the tf.data.Dataset objects, the training data is shuffled, so the following
# functions ensure that the correct (data, score) pairs get loaded into 
# the training and validation dataset objects.
##################################################################################
debug=False

def get_file_lengths(train_files):
    '''
    Returns the lengths of the files in train_files
    '''
    lengths = []
    for file in train_files:
        data = np.load(file, mmap_mode='r')
        lengths.append(len(data))
    return lengths

def compute_index_ranges(lengths):
    '''
    From the lengths given in [lengths], compute the corresponding index ranges
    for the target scores.
    '''
    ranges = []
    start = 0
    for l in lengths:
        end = start + l
        ranges.append((start, end))
        start = end
    return ranges

def load_train_with_targets(train_path, start_idx, end_idx, transform, all_targets):
    '''
    Load the training data at [train_path] and apply the [transform] function. Return
    the correct set of targets from [all_targets] based on the corresponding indices
    defined from [start_idx] and [end_idx].
    '''
    if debug:
        print("train path:", train_path)
        print("start, end:",start_idx,end_idx)
    data = transform(np.load(train_path.numpy().decode("utf-8")))  # shape (N, H, W, 1)
    targets = all_targets[start_idx:end_idx]
    if debug:
        plt.imshow(np.average(data,axis=0))
        plt.colorbar()
        plt.title("Average event")
        plt.show()

        plt.hist(targets,bins=30,histtype='step')
        plt.title("Targets")
        plt.show()
    return data, targets

def tf_load_file_with_targets(train_path, start_idx, end_idx, transform, all_targets):
    '''
    Wrap the load_train_with_targets function with a tf.py_function wrapper
    '''
    def wrapper(train_path, start_idx, end_idx):
        return load_train_with_targets(train_path, start_idx, end_idx, transform, all_targets)

    data, target = tf.py_function(
        func=wrapper,
        inp=[train_path, start_idx, end_idx],
        Tout=(tf.float32, tf.float32)
    )
    data.set_shape([None, 24, 24, 1])
    target.set_shape([None])
    return tf.data.Dataset.from_tensor_slices((data, target))

def get_train_ds(train_path_glob, all_targets, transform=lambda x:x):
    '''
    Create the train tf.data.Dataset that contains the pairs of training data and targets
    from the training data at [train_path_glob] and the target scores from [all_targets].
    '''
    print("debug",debug)
    BATCH_SIZE = 32
    # Calculate the index ranges for each data file
    train_files = sorted(glob.glob(train_path_glob))
    lengths = get_file_lengths(train_files)
    ranges = compute_index_ranges(lengths)

    assert sum(lengths) == len(all_targets), "Mismatch between data and targets!"

    # Combine the train files and data start, end indices
    train_info = list(zip(train_files, ranges))  # [(file_path, (start, end))]

    # Convert to dataset
    def gen():
        for path, (start, end) in train_info:
            yield path.encode('utf-8'), start, end

    dataset = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(), dtype=tf.string),
            tf.TensorSpec(shape=(), dtype=tf.int32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        )
    )

    # Broadcast the target array into tf.py_function using a closure
    def dataset_map(path, start_idx, end_idx):
        return tf_load_file_with_targets(path, start_idx, end_idx, transform, all_targets)

    # Create the train_ds
    train_ds = (
        dataset
        .shuffle(len(train_info))  # Shuffle file order
        .interleave(
            dataset_map,
            cycle_length=8,
            num_parallel_calls=tf.data.AUTOTUNE
        )
        .shuffle(1000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    return train_ds

def get_val_ds(val_path_glob, val_targets, transform=lambda x:x):
    '''
    Create the validation tf.data.Dataset that contains the pairs of validation data and targets
    from the validation data at [val_path_glob] and the target scores from [val_targets].
    '''
    val_files = sorted(glob.glob(val_path_glob))
    val_data = []
    for file in val_files:
        if debug:
            print(file)
        val_data.extend(transform(np.load(file)))

    assert len(val_data) == len(val_targets)

    val_ds = tf.data.Dataset.from_tensor_slices((val_data, val_targets))
    return val_ds.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

##################################################################################

def train_students_all_targets(label,teacher_path,target_dict,student_constructor,transform=lambda x:x,epochs=300):
    '''
    Trains a separate student model created by [student_constructor] for each target score
    in [target_dict] produced from the model at [teacher_path].

    Args:
        - label (str) : label for saving the student model and plots
        - teacher_path (str) : the path to the teacher model, used for labelling the saved student model and plots
        - target_dict : a dictionary outputted from get_train_test_val_target_dict that contains
                        the 'train_targets' and 'val_targets' for training the student model for each score
        - student_constructor : a function that returns (student_model, student_model_name) for training
        - transform : the transformation for the data
        - epochs : the default number of epochs to train
    '''
    dir = teacher_path[:teacher_path.rfind("/")]

    print("teacher path:", teacher_path)
    
    #print(save_name)

    jet_g_test = np.load("../jet_data/test/jetImages_g_0_cropped.npy")
    jet_q_test = np.load("../jet_data/test/jetImages_q_0_cropped.npy")
    jet_test = transform(np.concatenate((jet_g_test,jet_q_test)))
    print("test shape",jet_test.shape)

    signals = get_signal_dict(4,transform)

    roc_data_all_targets = {}

    for key in target_dict['train_targets'].keys():
        student_model,student_name = student_constructor()
        student_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-3), loss=keras.losses.MeanSquaredError(),metrics=['mse'])

        print("training on: ",key)
        train_ds_targets = get_train_ds("../jet_data/train/*",target_dict['train_targets'][key],transform)
        val_ds_targets = get_val_ds("../jet_data/val/*",target_dict['val_targets'][key],transform)
        save_name = f"{dir}/{student_name}_{label}_target{key}"        
        print(save_name)
        model_history, roc_data =  train_student_model(save_name,
                            model= student_model,
                            train_ds = train_ds_targets,
                            val_ds = val_ds_targets,
                            test=jet_test,
                            signals=signals,
                            epochs=epochs)
        roc_data_all_targets[f"target{key}"] = roc_data

    roc_data_all_targets_lists = recursive_dict_conversion_tolist(roc_data_all_targets)
    with open(f"{dir}/{student_name}_{label}_roc_data.json", "w") as f:
        json.dump(roc_data_all_targets_lists, f, indent=4) # indent for pretty printing

    teacher_json_file = f"{teacher_path[:teacher_path.rfind('.keras')]}_all_roc_data.json"
    print(teacher_json_file)
    with open(teacher_json_file, 'r') as file:
        teacher_json = json.load(file) 

    fig,axs = plt.subplots(2,5,figsize=(20,8),layout='constrained')
    for i,target in enumerate(roc_data_all_targets.keys()):
        print(target)
        for j,signal in enumerate(['w','z','t','combined_signal']):
            student_fpr = roc_data_all_targets[target]['fprs'][signal]
            student_tpr = roc_data_all_targets[target]['tprs'][signal]
    
            teacher_fpr = teacher_json[target]['fprs'][signal]
            teacher_tpr = teacher_json[target]['tprs'][signal]
    
            if signal=='combined_signal':
                axs[1,i].plot(teacher_fpr,teacher_tpr,color='tab:blue',linestyle='solid',label=f'teacher - {signal}')
                axs[1,i].plot(student_fpr,student_tpr,color="tab:blue",linestyle='dashed',label=f'student - {signal}')
                axs[1,i].set_title(f"{target} - combined signal")
            else:
                axs[0,i].plot(teacher_fpr,teacher_tpr,color=f'C{j}',linestyle='solid',label=f'teacher - {signal}')
                axs[0,i].plot(student_fpr,student_tpr,color=f'C{j}',linestyle='dashed',label=f'student - {signal}')
                axs[0,i].set_title(f"{target} - all signals")
    
            if i==4:
                axs[1,i].legend(bbox_to_anchor=(1.05,0.95))
                axs[0,i].legend(bbox_to_anchor=(1.05,0.95))
        axs[1,i].plot([0,1],[0,1],label='random guess',color='grey',linestyle='dashed')
        axs[0,i].plot([0,1],[0,1],label='random guess',color='grey',linestyle='dashed')
    
        axs[0,i].set_xscale('log')
        axs[1,i].set_xscale('log')
        axs[0,i].set_yscale('log')
        axs[1,i].set_yscale('log')
    
        axs[1,i].set_xlabel('FPR')
        axs[0,i].set_xlabel('FPR')
    
        axs[0,0].set_ylabel('TPR')
        axs[1,0].set_ylabel('TPR')
    plt.suptitle(f"Teacher and student comparison \n {label}")
    plt.savefig(f"{dir}/{student_name}_{label}_student_teacher_ROC.png")
    plt.show()
    return

# Train students on chosen teacher models

Train each of the five student architectures on each score target from each chosen teacher.

In [ ]:
teacher_paths = [ 'log_vae_4layer_small_latent3_lastfilter4_beta1e-4/log_vae_4layer_small_latent3_lastfilter4_beta1e-4.keras",
                 "clipped_vae_3layer_latent6_lastfilter4_beta1e-2/clipped_vae_3layer_latent6_lastfilter4_beta1e-2.keras",
                 "divided_vae_3layer_latent6_lastfilter4_beta1e-6/divided_vae_3layer_latent6_lastfilter4_beta1e-6.keras"]

transforms = [lambda x:np.log(x+1),
              lambda x:np.clip(x,None,72),
              lambda x:x/512]
betas = [1e-4,1e-2,1e-6]
for i,teacher_path in enumerate(teacher_paths):
    label = teacher_path[:teacher_path.find("_lastfilter")]
    dir = teacher_path[:teacher_path.find("/")]

    transform = transforms[i]
    beta = betas[i]

    target_dict = get_train_test_val_target_dict(f"{dir}/{label}",
                                                 teacher_path,
                                                "jet_data/",
                                                beta,
                                                transform)
    
    for student_constructor in [get_student_tiny,get_student_small,get_student_medium,get_student_large]:
        train_students_all_targets(label,
                                   teacher_path,
                                   target_dict,
                                   student_constructor,
                                   transform,
                                   epochs=600)